# GUI-Model: Mobile UI Transition & Action Predictor

Qwen3-VL-8B-Instruct 기반 모바일 UI 예측 모델 Fine-tuning 파이프라인.

**전체 파이프라인:**
1. **Stage 1: World Modeling** - UI State (XML) + Action → Next UI State (XML)
2. **Stage 2: Action Prediction** - Screenshot + UI State + Task → Action (JSON)
3. **Evaluation** - Test set으로 2-Way 모델 비교 평가
4. **Merge & Upload** - Best 모델 Merge 후 HuggingFace Hub 업로드

**모델 정보:**

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Template | qwen3_vl_nothink |
| Vision Tower | Frozen |

**데이터셋 정보:**

| Dataset | Stage | Entries | Description |
|---------|-------|---------|-------------|
| GUI-Model_stage1_train | 1 | ~2,988 | World Modeling (Train 95%) |
| GUI-Model_stage1_test | 1 | ~157 | World Modeling (Test 5%) |
| GUI-Model_stage2_train | 2 | ~3,290 | Action Prediction (Train 90%) |
| GUI-Model_stage2_test | 2 | ~365 | Action Prediction (Test 10%) |
| Images | - | 3,655 | Mobile UI Screenshots (PNG, 공유) |

**Stage 1 학습 실험:**

| ID | Method | LR Schedule | Learning Rate | Config |
|----|--------|-------------|---------------|--------|
| [1] | Full FT | constant | 2e-5 | stage1_full/qwen3_vl_8b_gui.yaml |

**Stage 2 학습 실험 (2-Way 비교):**

| ID | Base Model | HuggingFace ID | Config |
|----|------------|----------------|--------|
| [S2-1] | Qwen3-VL-8B (Baseline) | `Qwen/Qwen3-VL-8B-Instruct` | stage2_lora/baseline.yaml |
| [S2-2] | gWorld-8B | `trillionlabs/gWorld-8B` | stage2_lora/gworld.yaml |

**Stage 2 공통 설정:** LoRA r=16, α=32, dropout=0.1, LR=5e-5 (cosine), batch=4x2x2GPU=16, A100x2

**평가 메트릭:**

| Metric | Stage | Description |
|--------|-------|-------------|
| eval_loss / Perplexity | 1 | Next token prediction loss |
| BLEU / ROUGE-L | 1 | 생성 XML vs GT XML 유사도 |
| Element Accuracy | 1 | XML 요소(태그+속성) 일치율 |
| Type Accuracy | 2 | Action type 일치율 |
| Bounds IoU | 2 | Bounding box IoU |
| Params Match | 2 | 파라미터 정확도 |
| Overall Score | 2 | Type_Acc × (0.5×IoU + 0.5×Params) |

## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive/')

In [ ]:
import os

# ============================================================
# === 여기서 프로젝트 루트 경로를 설정하세요 ===
# BASE_DIR = "/path/to/GUI-Model"
BASE_DIR = "/Users/bsw/Library/Mobile Documents/com~apple~CloudDocs/Project/2026_GUI-Model/GUI-Model"
# ============================================================

LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")
DATA_DIR = os.path.join(BASE_DIR, "data")

os.chdir(BASE_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/hiyouga/LlamaFactory.git

In [ ]:
os.chdir(LF_ROOT)
!pip install -e ".[torch,metrics]"

## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 후 LLaMA-Factory에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: 3,655개 모바일 UI 스크린샷

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage1.jsonl | 3,145 | World Modeling 원본 (Filtered) |
| gui-model_stage1_train.jsonl | ~2,988 | Train Split (95%) |
| gui-model_stage1_test.jsonl | ~157 | Test Split (5%) |

**Split 전략:**
- Random Split (seed=42, 재현 가능)
- 비율: 0.95 : 0.05

In [ ]:
import json
import random
import shutil
from pathlib import Path

# === Paths ===
DATA_PATH = Path(DATA_DIR)
LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. 데이터 디렉토리 생성 ===
LF_GUI_DATA.mkdir(parents=True, exist_ok=True)
(LF_GUI_DATA / "images").mkdir(exist_ok=True)

# === 2. Train/Test Split (0.95:0.05) ===
stage1_path = DATA_PATH / "gui-model_stage1.jsonl"
train_path = DATA_PATH / "gui-model_stage1_train.jsonl"
test_path = DATA_PATH / "gui-model_stage1_test.jsonl"

with open(stage1_path, 'r') as f:
    entries = [json.loads(line) for line in f if line.strip()]

random.seed(42)
random.shuffle(entries)
split_idx = int(len(entries) * 0.95)
train_entries = entries[:split_idx]
test_entries = entries[split_idx:]

for path, data in [(train_path, train_entries), (test_path, test_entries)]:
    with open(path, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"[Split] Total: {len(entries)} → Train: {len(train_entries)}, Test: {len(test_entries)}")
print(f"  Ratio: {len(train_entries)/len(entries):.3f} : {len(test_entries)/len(entries):.3f}")

# === 3. JSONL 파일 복사 ===
for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        with open(src, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[Copy] {fname} ({count} entries, {src.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"[WARN] Not found: {src}")

# === 4. 이미지 복사 ===
src_images = DATA_PATH / "images"
dst_images = LF_GUI_DATA / "images"

if src_images.exists():
    img_count = len(list(src_images.glob("*.png")))
    existing = len(list(dst_images.glob("*.png")))
    if existing >= img_count:
        print(f"[Skip] Images already exist ({existing} files)")
    else:
        for img in src_images.glob("*.png"):
            dst_img = dst_images / img.name
            if not dst_img.exists():
                shutil.copy2(img, dst_img)
        print(f"[Copy] {img_count} images")
else:
    print(f"[WARN] Image directory not found: {src_images}")

# === 5. dataset_info.json 등록 ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

DATASETS_TO_REGISTER = {
    "GUI-Model_stage1_train": {
        "file_name": "GUI-Model/gui-model_stage1_train.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    },
    "GUI-Model_stage1_test": {
        "file_name": "GUI-Model/gui-model_stage1_test.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    }
}

for name, config in DATASETS_TO_REGISTER.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 1 Dataset Statistics ===\n")

for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()
        entry_count = len(lines)

        sample = json.loads(lines[0])
        msg_count = len(sample["messages"])
        img_count = len(sample.get("images", []))

        print(f"[{fname}]")
        print(f"  Entries: {entry_count}")
        print(f"  Sample - Messages: {msg_count}, Images: {img_count}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
        print()

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 변환, **Train/Test Split**, LLaMA-Factory 등록을 수행합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1과 동일한 3,655개 (공유)

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage2.jsonl | 3,655 | Action Prediction (원본) |
| gui-model_stage2_train.jsonl | ~3,290 | Train Split (90%) |
| gui-model_stage2_test.jsonl | ~365 | Test Split (10%) |

**변환 작업:**
- 원본의 `id`, `assets`, `source` 필드 제거
- `images` 필드 추가: `["GUI-Model/images/{id}.png"]`

**Split 전략:**
- Stratified Split (action type 비율 유지)
- Random seed: 42 (재현 가능)

In [ ]:
import json
from pathlib import Path

stage2_path = Path(DATA_DIR) / "gui-model_stage2.jsonl"

# === 1. 변환 필요 여부 확인 ===
with open(stage2_path, 'r') as f:
    first = json.loads(f.readline())

if "images" in first and "id" not in first:
    print("[Skip] Stage 2 data already converted")
    print(f"  Keys: {list(first.keys())}")
    print(f"  Images: {first['images']}")
else:
    # === 2. 변환 수행 ===
    with open(stage2_path, 'r') as f:
        lines = f.readlines()

    print(f"[Before] {len(lines)} entries, Keys: {list(first.keys())}")

    converted = []
    for line in lines:
        entry = json.loads(line)
        new_entry = {
            "messages": entry["messages"],
            "images": [f"GUI-Model/images/{entry['id']}.png"]
        }
        converted.append(json.dumps(new_entry, ensure_ascii=False))

    with open(stage2_path, 'w') as f:
        f.write('\n'.join(converted) + '\n')

    # === 3. 검증 ===
    with open(stage2_path, 'r') as f:
        check = json.loads(f.readline())

    print(f"[After] {len(converted)} entries, Keys: {list(check.keys())}")
    print(f"  Sample images: {check['images']}")
    print(f"  File size: {stage2_path.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict, Counter

stage2_path = Path(DATA_DIR) / "gui-model_stage2.jsonl"

# === 1. 전체 데이터 로드 ===
with open(stage2_path, 'r') as f:
    entries = [json.loads(line) for line in f if line.strip()]

print(f"[Load] Total entries: {len(entries)}")

# === 2. Action type별 그룹화 (Stratified) ===
type_groups = defaultdict(list)
for entry in entries:
    action = json.loads(entry['messages'][-1]['value'])
    action_type = action.get('type', 'unknown')
    type_groups[action_type].append(entry)

print(f"[Group] Action types: {dict(Counter({k: len(v) for k, v in type_groups.items()}))}")

# === 3. Stratified Split (0.9:0.1) ===
random.seed(42)
train_entries, test_entries = [], []

for action_type, group in type_groups.items():
    random.shuffle(group)
    split_idx = max(1, int(len(group) * 0.9))  # 최소 1개는 test에
    train_entries.extend(group[:split_idx])
    test_entries.extend(group[split_idx:])

# 셔플
random.shuffle(train_entries)
random.shuffle(test_entries)

# === 4. 저장 ===
train_path = Path(DATA_DIR) / "gui-model_stage2_train.jsonl"
test_path = Path(DATA_DIR) / "gui-model_stage2_test.jsonl"

for path, data in [(train_path, train_entries), (test_path, test_entries)]:
    with open(path, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

# === 5. 검증: action type 비율 비교 ===
print(f"\n{'='*60}")
print(f"Total: {len(entries)} -> Train: {len(train_entries)}, Test: {len(test_entries)}")
print(f"Split ratio: {len(train_entries)/len(entries):.3f} : {len(test_entries)/len(entries):.3f}")

for split_name, split_data in [("Train", train_entries), ("Test", test_entries)]:
    types = Counter()
    for e in split_data:
        a = json.loads(e['messages'][-1]['value'])
        types[a['type']] += 1
    print(f"\n[{split_name}] Action Type Distribution:")
    for t, c in types.most_common():
        print(f"  {t}: {c} ({c/len(split_data)*100:.1f}%)")

In [ ]:
import json
import shutil
from pathlib import Path

DATA_PATH = Path(DATA_DIR)
LF_DATA_DIR = Path(LF_ROOT) / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. Stage 2 JSONL 복사 (원본 + train + test) ===
for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        with open(src, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[Copy] {fname} ({count} entries, {src.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"[WARN] Not found: {src}")

# === 2. dataset_info.json에 Stage 2 등록 (train + test) ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"
with open(dataset_info_path, 'r', encoding='utf-8') as f:
    dataset_info = json.load(f)

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

STAGE2_DATASETS = {
    "GUI-Model_stage2_train": {
        "file_name": "GUI-Model/gui-model_stage2_train.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    },
    "GUI-Model_stage2_test": {
        "file_name": "GUI-Model/gui-model_stage2_test.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    }
}

for name, config in STAGE2_DATASETS.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 2 Dataset Statistics ===\n")

for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()

        print(f"[{fname}]")
        print(f"  Entries: {len(lines)}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

        # Action type 분포
        action_types = Counter()
        for line in lines:
            entry = json.loads(line)
            gpt_val = json.loads(entry['messages'][-1]['value'])
            action_types[gpt_val.get('type', 'unknown')] += 1

        print(f"  Action Type Distribution:")
        for atype, count in action_types.most_common():
            print(f"    {atype}: {count} ({count/len(lines)*100:.1f}%)")
        print()
    else:
        print(f"[WARN] Not found: {fpath}")

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 3. Stage 1 SFT Training

Qwen3-VL-8B-Instruct를 Stage 1 (World Modeling) 데이터로 Full Fine-tuning합니다.
DeepSpeed ZeRO-3 필수 (2x A100 이상 권장).

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Method | Full (all parameters) |
| Dataset | GUI-Model_stage1_train (~2,988건, 95% split) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 8,192 |
| Epochs | 3.0 |
| Learning Rate | 2e-5 |
| LR Scheduler | constant |
| Warmup | 0.0 |
| Effective Batch | 16 (1 x 8 x 2 GPU) |
| Precision | bf16 |
| Vision Tower | Frozen |
| DeepSpeed | ZeRO-3 |
| Gradient Checkpointing | Enabled |
| Weight Decay | 0.01 |
| Hardware | A100 80GB x 2 |
| Save Steps | 50 |
| Eval Steps | 50 |

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage1_full"
yaml_dir.mkdir(parents=True, exist_ok=True)

STAGE1_CONFIG = """\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_train
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_full
logging_steps: 1
save_steps: 50
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-5
num_train_epochs: 3.0
lr_scheduler_type: constant
warmup_ratio: 0.0
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
deepspeed: examples/deepspeed/ds_z3_config.json

# ### eval
# val_size: 0.1
# per_device_eval_batch_size: 1
# eval_strategy: steps
# eval_steps: 50
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## [1] Full Fine-tuning (constant, lr=2e-5, DeepSpeed Z3)
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage1_full/qwen3_vl_8b_gui.yaml

## 4. Stage 1 Evaluation Pipeline

학습된 Stage 1 모델의 World Modeling 성능을 평가합니다.

**평가 방법:**

| Method | Metric | Description |
|--------|--------|-------------|
| Loss 기반 | eval_loss | Test set에 대한 Cross-entropy loss |
| Loss 기반 | Perplexity | exp(eval_loss) - 낮을수록 좋음 |
| 생성 비교 | BLEU-4 | n-gram 기반 텍스트 유사도 |
| 생성 비교 | ROUGE-L | Longest Common Subsequence F1 |
| 생성 비교 | Exact Match | GT XML과 완전 일치 비율 |
| 생성 비교 | Tag Accuracy | XML 태그 시퀀스 위치별 일치율 |
| 생성 비교 | Index Coverage | GT XML의 index 속성 커버리지 |

**데이터:** GUI-Model_stage1_test (~157건, 5% split)

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage1_eval"
yaml_dir.mkdir(parents=True, exist_ok=True)

# === 1. Loss 기반 평가 YAML ===
EVAL_LOSS_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_eval: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_test
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_eval_loss
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 1
"""

fpath = yaml_dir / "eval_loss.yaml"
with open(fpath, 'w') as f:
    f.write(EVAL_LOSS_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

# === 2. 생성 비교 평가 YAML ===
PREDICT_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_predict: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_test
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_predict
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

fpath = yaml_dir / "predict.yaml"
with open(fpath, 'w') as f:
    f.write(PREDICT_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## Stage 1 - Loss 기반 평가 (eval_loss / perplexity)
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage1_eval/eval_loss.yaml

In [ ]:
os.chdir(LF_ROOT)

## Stage 1 - 생성 비교 평가 (Prediction)
!python scripts/vllm_infer.py \
    --model_name_or_path ./outputs/stage1_full \
    --dataset GUI-Model_stage1_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage1_predict/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage1_predict/predict_results.json

In [ ]:
import json
import re
import math
from collections import Counter
from xml.etree import ElementTree as ET

def parse_xml_elements(xml_str):
    """XML 문자열에서 요소 목록을 추출합니다."""
    try:
        xml_str = xml_str.strip()
        if not xml_str.startswith('<'):
            return []
        # self-closing 태그 정규화
        xml_clean = re.sub(r'<(\w+)([^>]*)/>', r'<\1\2></\1>', xml_str)
        root = ET.fromstring(f"<root>{xml_clean}</root>")
        elements = []
        for elem in root.iter():
            if elem.tag == 'root':
                continue
            elements.append({
                'tag': elem.tag,
                'attrib': dict(elem.attrib),
                'text': (elem.text or '').strip(),
            })
        return elements
    except ET.ParseError:
        return []

def calc_element_accuracy(gt_xml, pred_xml):
    """XML 요소 수준의 정확도를 계산합니다."""
    gt_elements = parse_xml_elements(gt_xml)
    pred_elements = parse_xml_elements(pred_xml)

    if not gt_elements:
        return {'tag_accuracy': 1.0 if not pred_elements else 0.0,
                'index_coverage': 1.0 if not pred_elements else 0.0,
                'gt_count': 0, 'pred_count': 0}

    # Tag sequence matching
    gt_tags = [e['tag'] for e in gt_elements]
    pred_tags = [e['tag'] for e in pred_elements]
    matches = sum(1 for g, p in zip(gt_tags, pred_tags) if g == p)
    tag_accuracy = matches / len(gt_tags)

    # Index coverage
    gt_indices = {e['attrib'].get('index') for e in gt_elements if 'index' in e['attrib']}
    pred_indices = {e['attrib'].get('index') for e in pred_elements if 'index' in e['attrib']}
    index_coverage = len(gt_indices & pred_indices) / len(gt_indices) if gt_indices else 1.0

    return {'tag_accuracy': tag_accuracy, 'index_coverage': index_coverage,
            'gt_count': len(gt_elements), 'pred_count': len(pred_elements)}

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)

def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        elem_acc = calc_element_accuracy(gt_text, pred_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'element_accuracy': elem_acc,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_tag_accuracy': sum(r['element_accuracy']['tag_accuracy'] for r in results) / total if total else 0,
        'avg_index_coverage': sum(r['element_accuracy']['index_coverage'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, calc_element_accuracy, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

test_path = Path(LF_ROOT) / "data" / "GUI-Model" / "gui-model_stage1_test.jsonl"
pred_path = Path(LF_ROOT) / "outputs" / "stage1_predict" / "generated_predictions.jsonl"

# === 1. Loss 기반 결과 ===
loss_result_path = Path(LF_ROOT) / "outputs" / "stage1_eval_loss" / "eval_results.json"
if loss_result_path.exists():
    with open(loss_result_path, 'r') as f:
        loss_metrics = json.load(f)
    eval_loss = loss_metrics.get('eval_loss', 0)
    print("=== Loss-based Evaluation ===")
    print(f"  eval_loss:   {eval_loss:.4f}")
    print(f"  perplexity:  {math.exp(eval_loss):.4f}")
    print()
else:
    print("[SKIP] Loss evaluation results not found\n")

# === 2. 생성 비교 결과 ===
if pred_path.exists():
    metrics, results = evaluate_stage1_predictions(str(test_path), str(pred_path))

    print("=== Generation-based Evaluation ===")
    summary = pd.DataFrame({
        'Metric': ['BLEU-4', 'ROUGE-L', 'Exact Match', 'Tag Accuracy', 'Index Coverage'],
        'Score': [
            f"{metrics['avg_bleu']:.4f}",
            f"{metrics['avg_rouge_l']:.4f}",
            f"{metrics['exact_match_rate']:.1%}",
            f"{metrics['avg_tag_accuracy']:.4f}",
            f"{metrics['avg_index_coverage']:.4f}",
        ]
    })
    display(summary)

    # 샘플 비교 (첫 3건)
    with open(test_path, 'r') as f:
        gt_samples = [json.loads(line) for line in f][:3]
    with open(pred_path, 'r') as f:
        pred_samples = [json.loads(line) for line in f][:3]

    print("\n=== Sample Comparisons (first 3) ===")
    for i, (gt, pred) in enumerate(zip(gt_samples, pred_samples)):
        gt_text = gt['messages'][-1]['value']
        pred_text = pred.get('predict', pred.get('output', ''))
        print(f"\n--- Sample {i+1} ---")
        print(f"  BLEU: {results[i]['bleu']:.4f}, ROUGE-L: {results[i]['rouge_l']:.4f}")
        print(f"  GT (first 200 chars):   {gt_text[:200]}...")
        print(f"  Pred (first 200 chars): {pred_text[:200]}...")
else:
    print("[SKIP] Prediction results not found. Run prediction cell first.")

## 5. Stage 1 Merge & Upload to HuggingFace

학습된 Stage 1 LoRA 어댑터를 기본 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**업로드 대상:** `SaFD-00/qwen3-vl-8b-gui`

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "gui"
yaml_dir.mkdir(parents=True, exist_ok=True)

MERGE_STAGE1_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full
trust_remote_code: true
template: qwen3_vl_nothink

### export
export_dir: ./exports/qwen3-vl-8b-gui
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: SaFD-00/qwen3-vl-8b-gui
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(MERGE_STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## LoRA Merge & HuggingFace Hub Upload
!llamafactory-cli export examples/merge_custom/GUI-Model/gui/qwen3_vl_8b_gui.yaml

## 6. Stage 2 SFT Training - World Model 사전학습 효과 검증

**핵심 가설**: GUI World Modeling으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT에서 더 빠르게 수렴할 것이다.

### 2-Way 비교 설계

| ID | 역할 | 모델 | HuggingFace ID |
|----|------|------|----------------|
| [S2-1] | Baseline | Qwen3-VL-8B-Instruct | `Qwen/Qwen3-VL-8B-Instruct` |
| [S2-2] | World Model | gWorld-8B | `trillionlabs/gWorld-8B` |

> 둘 모두 동일한 베이스 모델(Qwen3-VL-8B-Instruct)에서 파생. 공정한 비교 가능.

**공통 학습 설정:**

| 항목 | 값 |
|------|-----|
| Dataset | GUI-Model_stage2_train (~3,290건, 90% split) |
| Method | LoRA (rank=16, alpha=32, target=all, dropout=0.1) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 8,192 |
| Epochs | 1.0 |
| Learning Rate | 5e-5 (cosine, warmup=0.05) |
| Effective Batch | 16 (4 x 2 x 2 GPU) |
| Weight Decay | 0.01 |
| Logging Steps | 1 |
| Precision | bf16 |
| Vision Tower | Frozen |
| Hardware | A100 80GB x 2 |

In [ ]:
import os
from pathlib import Path

# === Stage 2 YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage2_lora"
yaml_dir.mkdir(parents=True, exist_ok=True)

# 공통 설정 템플릿
COMMON_CONFIG = """\
### model
model_name_or_path: {model_name_or_path}
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: true
lora_rank: 16
lora_alpha: 32
lora_target: all
lora_dropout: 0.1

### dataset
dataset: GUI-Model_stage2_train
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{output_dir}
logging_steps: 1
save_steps: 100
plot_loss: true
overwrite_output_dir: true
resume_from_checkpoint: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 2
learning_rate: 5.0e-5
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
"""

MODELS = {
    "baseline.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "output_dir": "stage2_baseline",
    },
    "gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "output_dir": "stage2_gworld",
    },
}

for fname, params in MODELS.items():
    content = COMMON_CONFIG.format(
        model_name_or_path=params["model_name_or_path"],
        output_dir=params["output_dir"],
    )
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  base: {params['model_name_or_path']}")
    print(f"  output: ./outputs/{params['output_dir']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [S2-1] Stage 2 - Baseline (Qwen3-VL-8B-Instruct) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/baseline.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-2] Stage 2 - gWorld-8B (World Model) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/gworld.yaml

## 7. Stage 2 Evaluation Pipeline

Test set(`GUI-Model_stage2_test`)을 사용하여 2개 모델의 Action Prediction 성능을 비교합니다.

**파이프라인:**
1. Prediction YAML 생성 (2모델)
2. `do_predict`로 test set prediction 수행
3. 평가 함수로 action 단위 메트릭 계산
4. 모델별 비교 테이블 & 시각화

**평가 메트릭:**

| Metric | Formula | Description |
|--------|---------|-------------|
| Parse Rate | 유효 JSON / 전체 | 모델 출력 파싱 성공률 |
| Type Accuracy | 정확 type / 전체 | Action type 일치율 |
| Bounds IoU | IoU(GT, Pred) | Bounding box 겹침 비율 |
| Params Match | 정확 params / 전체 | 파라미터 정확도 |
| Overall Score | Type_Acc × (0.5×IoU + 0.5×Params) | 종합 점수 |

In [ ]:
from pathlib import Path

# === Prediction YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage2_predict"
yaml_dir.mkdir(parents=True, exist_ok=True)

PREDICT_TEMPLATE = """\
### model
model_name_or_path: {model_name_or_path}
adapter_name_or_path: {adapter_path}
trust_remote_code: true
finetuning_type: lora
template: qwen3_vl_nothink
image_max_pixels: 4233600

### method
stage: sft
do_predict: true

### dataset
dataset: GUI-Model_stage2_test
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{output_dir}
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

PREDICT_MODELS = {
    "predict_baseline.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "adapter_path": "./outputs/stage2_baseline",
        "output_dir": "stage2_predict_baseline",
    },
    "predict_gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "adapter_path": "./outputs/stage2_gworld",
        "output_dir": "stage2_predict_gworld",
    },
}

for fname, params in PREDICT_MODELS.items():
    content = PREDICT_TEMPLATE.format(**params)
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  base: {params['model_name_or_path']}")
    print(f"  adapter: {params['adapter_path']}")
    print(f"  output: ./outputs/{params['output_dir']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [S2-1] Prediction - Baseline (Qwen3-VL-8B-Instruct) | A100 x 2

# !FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
#     llamafactory-cli train examples/train_custom/GUI-Model/stage2_predict/predict_baseline.yaml

!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --adapter_name_or_path ./outputs/stage2_baseline \
    --dataset GUI-Model_stage2_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage2_predict_baseline/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage2_predict_baseline/predict_results.json

In [ ]:
os.chdir(LF_ROOT)

## [S2-2] Prediction - gWorld-8B | A100 x 2

# !FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
#     llamafactory-cli train examples/train_custom/GUI-Model/stage2_predict/predict_gworld.yaml

!python scripts/vllm_infer.py \
    --model_name_or_path trillionlabs/gWorld-8B \
    --adapter_name_or_path ./outputs/stage2_gworld \
    --dataset GUI-Model_stage2_test \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name ./outputs/stage2_predict_gworld/generated_predictions.jsonl \
    --matrix_save_name ./outputs/stage2_predict_gworld/predict_results.json

In [ ]:
import json
import re
from collections import defaultdict

def parse_bounds(bounds_str):
    """Parse bounds string '[x1, y1, x2, y2]' to list of floats."""
    try:
        nums = re.findall(r'[\d.]+', str(bounds_str))
        return [float(n) for n in nums[:4]] if len(nums) >= 4 else None
    except:
        return None

def calc_iou(box1, box2):
    """Calculate IoU between two bounding boxes [x1, y1, x2, y2]."""
    if box1 is None or box2 is None:
        return 0.0
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def parse_action(text):
    """Parse action JSON from model output text."""
    text = text.strip()
    # Try direct JSON parse
    try:
        return json.loads(text)
    except:
        pass
    # Try extracting JSON from markdown code block
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass
    # Try finding first JSON object
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return None

def evaluate_single(gt_action, pred_action):
    """Evaluate a single prediction against ground truth."""
    result = {'type_correct': False, 'iou': 0.0, 'params_correct': False, 'has_bounds': False, 'has_params': False}

    if pred_action is None:
        return result

    gt_type = gt_action.get('type', '')
    pred_type = pred_action.get('type', '')
    result['type_correct'] = (gt_type.lower() == pred_type.lower())

    # Bounds IoU
    gt_bounds = parse_bounds(gt_action.get('bounds'))
    pred_bounds = parse_bounds(pred_action.get('bounds'))
    if gt_bounds is not None:
        result['has_bounds'] = True
        result['iou'] = calc_iou(gt_bounds, pred_bounds)

    # Params match
    param_keys = {'openapp': 'app', 'input': 'text', 'swipe': 'direction'}
    if gt_type.lower() in param_keys:
        result['has_params'] = True
        key = param_keys[gt_type.lower()]
        gt_val = str(gt_action.get(key, '')).strip().lower()
        pred_val = str(pred_action.get(key, '')).strip().lower()
        result['params_correct'] = (gt_val == pred_val)
    elif gt_type.lower() not in ['click', 'long click']:
        # Types without bounds or special params
        result['has_params'] = False

    return result

def evaluate_predictions(test_path, pred_path):
    """Evaluate all predictions and return metrics."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]

    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    per_type = defaultdict(lambda: {'count': 0, 'type_correct': 0, 'iou_sum': 0.0, 'iou_count': 0, 'params_correct': 0, 'params_count': 0})

    for i, (gt_entry, pred_entry) in enumerate(zip(gt_entries, pred_entries)):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        r['gt_type'] = gt_action.get('type', 'unknown')
        results.append(r)

        t = r['gt_type']
        per_type[t]['count'] += 1
        per_type[t]['type_correct'] += int(r['type_correct'])
        if r['has_bounds']:
            per_type[t]['iou_sum'] += r['iou']
            per_type[t]['iou_count'] += 1
        if r['has_params']:
            per_type[t]['params_correct'] += int(r['params_correct'])
            per_type[t]['params_count'] += 1

    total = len(results)
    parsed = sum(1 for r in results if r['type_correct'] or r.get('gt_type'))
    type_correct = sum(r['type_correct'] for r in results)
    bounds_entries = [r for r in results if r['has_bounds']]
    params_entries = [r for r in results if r['has_params']]

    avg_iou = sum(r['iou'] for r in results) / total if total else 0
    cond_iou = sum(r['iou'] for r in bounds_entries) / len(bounds_entries) if bounds_entries else 0
    params_acc = sum(r['params_correct'] for r in params_entries) / len(params_entries) if params_entries else 0
    cond_params = params_acc
    type_acc = type_correct / total if total else 0
    overall = type_acc * (0.5 * avg_iou + 0.5 * (sum(r['params_correct'] for r in results) / total if total else 0))

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count': d['count'],
            'type_acc': d['type_correct'] / d['count'] if d['count'] else 0,
            'avg_iou': d['iou_sum'] / d['iou_count'] if d['iou_count'] else 0,
            'params_acc': d['params_correct'] / d['params_count'] if d['params_count'] else 0,
        }

    metrics = {
        'total': total,
        'parse_rate': parsed / total if total else 0,
        'type_accuracy': type_acc,
        'avg_bounds_iou': avg_iou,
        'cond_bounds_iou': cond_iou,
        'params_accuracy': sum(r['params_correct'] for r in results) / total if total else 0,
        'cond_params_acc': cond_params,
        'overall_score': overall,
        'per_type': per_type_summary,
    }

    return metrics, results

print("[Loaded] Evaluation functions: parse_bounds, calc_iou, parse_action, evaluate_single, evaluate_predictions")

In [ ]:
import json
import pandas as pd
from pathlib import Path

test_path = Path(LF_ROOT) / "data" / "GUI-Model" / "gui-model_stage2_test.jsonl"

MODEL_PREDS = {
    "Baseline (Qwen3-VL-8B)": Path(LF_ROOT) / "outputs" / "stage2_predict_baseline" / "generated_predictions.jsonl",
    "gWorld-8B": Path(LF_ROOT) / "outputs" / "stage2_predict_gworld" / "generated_predictions.jsonl",
}

all_metrics = {}
all_results = {}

for model_name, pred_path in MODEL_PREDS.items():
    if not pred_path.exists():
        print(f"[SKIP] {model_name}: {pred_path.name} not found")
        continue
    metrics, results = evaluate_predictions(str(test_path), str(pred_path))
    all_metrics[model_name] = metrics
    all_results[model_name] = results
    print(f"[Done] {model_name}")

# === Summary Table ===
if all_metrics:
    summary_df = pd.DataFrame({
        name: {
            'Parse Rate': f"{m['parse_rate']:.1%}",
            'Type Accuracy': f"{m['type_accuracy']:.1%}",
            'Bounds IoU': f"{m['avg_bounds_iou']:.3f}",
            'Params Accuracy': f"{m['params_accuracy']:.1%}",
            'Cond. IoU': f"{m['cond_bounds_iou']:.3f}",
            'Cond. Params': f"{m['cond_params_acc']:.1%}",
            'Overall Score': f"{m['overall_score']:.3f}",
        }
        for name, m in all_metrics.items()
    }).T

    print("\n" + "="*70)
    print("=== Stage 2 Model Comparison ===")
    print("="*70)
    display(summary_df)

    # === Per-Type Table ===
    for model_name, m in all_metrics.items():
        print(f"\n--- {model_name} Per-Type ---")
        type_df = pd.DataFrame(m['per_type']).T
        type_df.columns = ['Count', 'Type Acc', 'Avg IoU', 'Params Acc']
        display(type_df)
else:
    print("[WARN] No prediction results found. Run prediction cells first.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_names = list(all_metrics.keys())
metrics_to_plot = ['type_accuracy', 'avg_bounds_iou', 'params_accuracy', 'overall_score']
metric_labels = ['Type Accuracy', 'Bounds IoU', 'Params Accuracy', 'Overall Score']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# === 1. Bar Chart: 모델별 메트릭 비교 ===
x = np.arange(len(metrics_to_plot))
width = 0.3
colors = ['#2196F3', '#4CAF50']

for i, name in enumerate(model_names):
    values = [all_metrics[name][m] for m in metrics_to_plot]
    axes[0].bar(x + i * width, values, width, label=name, color=colors[i])

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Stage 2: Model Comparison')
axes[0].set_xticks(x + width / 2)
axes[0].set_xticklabels(metric_labels, rotation=15)
axes[0].legend()
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', alpha=0.3)

# === 2. Per-Type Accuracy Comparison ===
action_types = sorted(set(
    t for m in all_metrics.values() for t in m['per_type'].keys()
))
x2 = np.arange(len(action_types))

for i, name in enumerate(model_names):
    values = [all_metrics[name]['per_type'].get(t, {}).get('type_acc', 0) for t in action_types]
    axes[1].bar(x2 + i * width, values, width, label=name, color=colors[i])

axes[1].set_xlabel('Action Type')
axes[1].set_ylabel('Type Accuracy')
axes[1].set_title('Per-Type Accuracy Comparison')
axes[1].set_xticks(x2 + width / 2)
axes[1].set_xticklabels(action_types, rotation=30)
axes[1].legend()
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LF_ROOT, 'outputs', 'stage2_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n[Saved] {os.path.join(LF_ROOT, 'outputs', 'stage2_evaluation.png')}")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def generate_eval_report(model_name, metrics, results, output_dir, comparison_metrics=None):
    """모델별 평가 결과를 Markdown 리포트로 생성하여 저장합니다."""
    m = metrics
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    lines = []
    lines.append(f"# Stage 2 Evaluation Report: {model_name}")
    lines.append(f"\n> Generated: {now}\n")

    # --- 실험 설정 ---
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Model | {model_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Task | Action Prediction (Screenshot + XML + Task → Action JSON) |")
    lines.append(f"| Method | LoRA (r=16, α=32, dropout=0.1) |")
    lines.append(f"| Training | 1 epoch, LR=5e-5 (cosine), batch=16, A100x2 |")
    lines.append("")

    # --- Overall Metrics ---
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    lines.append(f"| Parse Rate | {m['parse_rate']:.1%} |")
    lines.append(f"| Type Accuracy | {m['type_accuracy']:.1%} |")
    lines.append(f"| Bounds IoU (avg) | {m['avg_bounds_iou']:.4f} |")
    lines.append(f"| Bounds IoU (conditional) | {m['cond_bounds_iou']:.4f} |")
    lines.append(f"| Params Accuracy (avg) | {m['params_accuracy']:.1%} |")
    lines.append(f"| Params Accuracy (conditional) | {m['cond_params_acc']:.1%} |")
    lines.append(f"| **Overall Score** | **{m['overall_score']:.4f}** |")
    lines.append("")

    # --- Per-Type Breakdown ---
    lines.append("## Per-Type Breakdown\n")
    lines.append("| Action Type | Count | Type Acc | Avg IoU | Params Acc |")
    lines.append("|-------------|-------|----------|---------|------------|")
    for t, d in sorted(m['per_type'].items(), key=lambda x: -x[1]['count']):
        lines.append(f"| {t} | {d['count']} | {d['type_acc']:.1%} | {d['avg_iou']:.4f} | {d['params_acc']:.1%} |")
    lines.append("")

    # --- Comparison (있을 경우) ---
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")

        metric_rows = [
            ("Parse Rate", "parse_rate", ".1%"),
            ("Type Accuracy", "type_accuracy", ".1%"),
            ("Bounds IoU", "avg_bounds_iou", ".4f"),
            ("Cond. IoU", "cond_bounds_iou", ".4f"),
            ("Params Accuracy", "params_accuracy", ".1%"),
            ("Cond. Params", "cond_params_acc", ".1%"),
            ("Overall Score", "overall_score", ".4f"),
        ]
        for label, key, fmt in metric_rows:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")

    # --- Metric Definitions ---
    lines.append("## Metric Definitions\n")
    lines.append("| Metric | Description |")
    lines.append("|--------|-------------|")
    lines.append("| Parse Rate | 모델 출력이 유효한 JSON으로 파싱된 비율 |")
    lines.append("| Type Accuracy | 예측된 action type이 GT와 일치하는 비율 |")
    lines.append("| Bounds IoU (avg) | 전체 샘플 대상 bounding box IoU 평균 (bounds 없는 타입은 0) |")
    lines.append("| Bounds IoU (conditional) | bounds가 있는 샘플만 대상 IoU 평균 |")
    lines.append("| Params Accuracy (avg) | 전체 샘플 대상 파라미터 정확도 (params 없는 타입은 0) |")
    lines.append("| Params Accuracy (conditional) | params가 있는 샘플만 대상 정확도 |")
    lines.append("| Overall Score | Type_Acc × (0.5 × Avg_IoU + 0.5 × Avg_Params) |")
    lines.append("")

    report = "\n".join(lines)

    # --- 저장 ---
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)

    return str(output_path)

# === 각 모델별 리포트 생성 및 저장 ===
OUTPUT_DIRS = {
    "Baseline (Qwen3-VL-8B)": Path(LF_ROOT) / "outputs" / "stage2_predict_baseline",
    "gWorld-8B": Path(LF_ROOT) / "outputs" / "stage2_predict_gworld",
}

for model_name, m in all_metrics.items():
    output_dir = OUTPUT_DIRS.get(model_name)
    if output_dir:
        saved = generate_eval_report(
            model_name=model_name,
            metrics=m,
            results=all_results[model_name],
            output_dir=output_dir,
            comparison_metrics=all_metrics,
        )
        print(f"[Saved] {saved}")

print("\nDone.")

## 8. Stage 2 Merge & Upload to HuggingFace

Stage 2 LoRA 어댑터를 베이스 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**Merge 대상:**

| ID | Base Model | Adapter | Export |
|----|------------|---------|--------|
| [S2-1] | `Qwen/Qwen3-VL-8B-Instruct` | `./outputs/stage2_baseline` | `SaFD-00/qwen3-vl-8b-gui-baseline` |
| [S2-2] | `trillionlabs/gWorld-8B` | `./outputs/stage2_gworld` | `SaFD-00/qwen3-vl-8b-gui-gworld` |

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

# === Stage 2 Merge YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "stage2"
yaml_dir.mkdir(parents=True, exist_ok=True)

MERGE_TEMPLATE = """\
### model
model_name_or_path: {model_name_or_path}
adapter_name_or_path: {adapter_path}
trust_remote_code: true
finetuning_type: lora
template: qwen3_vl_nothink

### export
export_dir: ./exports/{export_dir}
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {hub_model_id}
"""

MERGE_MODELS = {
    "merge_baseline.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "adapter_path": "./outputs/stage2_baseline",
        "export_dir": "qwen3-vl-8b-gui-baseline",
        "hub_model_id": "SaFD-00/qwen3-vl-8b-gui-baseline",
    },
    "merge_gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "adapter_path": "./outputs/stage2_gworld",
        "export_dir": "qwen3-vl-8b-gui-gworld",
        "hub_model_id": "SaFD-00/qwen3-vl-8b-gui-gworld",
    },
}

for fname, params in MERGE_MODELS.items():
    content = MERGE_TEMPLATE.format(**params)
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  base: {params['model_name_or_path']}")
    print(f"  adapter: {params['adapter_path']}")
    print(f"  export: {params['hub_model_id']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [S2-1] Merge - Baseline (Qwen3-VL-8B-Instruct)
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/merge_baseline.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-2] Merge - gWorld-8B
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/merge_gworld.yaml